# Step 3: logistic regression

Features: the last 5 daily returns and yesterday's direction. Each market runs through the six walk-forward folds (test years 2017–2022). Part A uses sklearn; Part B is the same model in PyTorch, checked against sklearn.

In [1]:
import numpy as np
import pandas as pd

from data import load_all
from models import fit_sklearn, fit_torch, pooled, report, walk_forward

markets = load_all()
C_GRID = [0.0001, 0.001, 0.01, 0.1, 1, 10]   # sklearn: smaller C = stronger penalty
WD_GRID = [0.0001, 0.001, 0.01, 0.1, 1, 10]  # PyTorch: larger weight decay = stronger penalty

## Part A: sklearn

Per test year for India (change the name to see another market):

In [2]:
sk = {market: walk_forward(df, fit_sklearn, C_GRID) for market, df in markets.items()}
report(sk["India"]).round(3)

,setting,accuracy %,MCC,AUC,log loss,fair bar %,p vs fair,stretch bar %,stretch guess,p vs stretch
year,,,,,,,,,,
2017,0.000,56.275,0.000,0.532,0.687,56.275,1.000,56.275,always up,1.000
2018,10.000,52.653,0.011,0.534,0.691,53.878,0.784,53.878,always up,0.784
2019,0.001,52.282,-0.031,0.534,0.691,52.697,1.000,53.112,same as yesterday,0.924
2020,0.001,56.800,-0.013,0.530,0.684,58.800,0.405,58.800,always up,0.405
2021,0.000,55.645,0.000,0.542,0.687,55.645,1.000,56.855,same as yesterday,0.850
2022,0.001,49.194,-0.052,0.580,0.692,50.000,0.688,59.274,same as yesterday,0.026


All six test years pooled, per market:

In [3]:
pd.DataFrame({market: pooled(p) for market, p in sk.items()}).T.round(3)

,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday
Indonesia,52.917,0.011,0.514,52.917,1.000,49.485,0.061
Malaysia,49.624,-0.007,0.483,49.761,0.913,49.282,0.879
Philippines,51.368,-0.042,0.509,51.778,0.146,48.427,0.112
Singapore,52.125,0.038,0.527,49.402,0.073,49.602,0.164
Thailand,52.093,-0.056,0.495,52.437,0.062,50.995,0.567
India,53.820,-0.009,0.537,54.564,0.278,54.767,0.593


## Part B: PyTorch

Takes about 2 minutes.

In [4]:
tc = {market: walk_forward(df, fit_torch, WD_GRID) for market, df in markets.items()}
pd.DataFrame({market: pooled(p) for market, p in tc.items()}).T.round(3)

,accuracy %,MCC,AUC,fair bar %,p vs fair,same as yesterday %,p vs same as yesterday
Indonesia,52.917,0.008,0.516,52.917,1.000,49.485,0.062
Malaysia,49.897,0.005,0.483,49.761,0.910,49.282,0.762
Philippines,51.847,0.027,0.505,51.778,1.000,48.427,0.065
Singapore,51.859,0.033,0.527,49.402,0.100,49.602,0.216
Thailand,52.093,-0.056,0.494,52.437,0.062,50.995,0.567
India,54.699,0.030,0.539,54.564,0.921,54.767,1.000


## Do the two versions agree?

Target: correlation above 0.99 in every fold, and almost all up/down calls the same.

In [5]:
rows = []
for market in markets:
    both = sk[market][["year", "p_up"]].join(tc[market]["p_up"].rename("p_up_torch"))
    for year, f in both.groupby("year"):
        rows.append({
            "market": market,
            "year": year,
            "correlation": np.corrcoef(f["p_up"], f["p_up_torch"])[0, 1],
            "calls agree %": 100 * ((f["p_up"] > 0.5) == (f["p_up_torch"] > 0.5)).mean(),
        })
agreement = pd.DataFrame(rows)
agreement.round(4)

,market,year,correlation,calls agree %
0,Indonesia,2017,0.9997,99.1597
1,Indonesia,2018,0.9998,98.7500
2,Indonesia,2019,0.9995,99.1837
3,Indonesia,2020,0.9999,98.7552
4,Indonesia,2021,1.0000,100.0000
5,Indonesia,2022,0.9933,96.7480
6,Malaysia,2017,1.0000,100.0000
7,Malaysia,2018,1.0000,100.0000
8,Malaysia,2019,0.9998,97.5410
9,Malaysia,2020,1.0000,100.0000


In [6]:
print(f"lowest correlation:     {agreement['correlation'].min():.4f}")
print(f"lowest calls agreement: {agreement['calls agree %'].min():.1f}%")

lowest correlation:     0.9933
lowest calls agreement: 86.7%
